## Car Parc Analysis

**Location:** 1303 Woodbury Ave, Portsmouth, NH  
**Coordinates:** latitude 43.0859778, longitude -70.78677

**Objective:** Pull car parc (total vehicles) data for drive times from 5-25 minutes in 1-minute increments

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='.env')

placer_api_key = os.getenv('PLACER_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if not placer_api_key:
    raise ValueError("PLACER_API_KEY not found in .env file")
if not google_api_key:
    raise ValueError("GOOGLE_API_KEY not found in .env file")

print("✓ API keys loaded successfully")

In [ ]:
import requests
import time
from datetime import datetime, timedelta
from pydantic import BaseModel
from typing import Optional, List, Dict, Any

print("✓ Imports loaded")

In [ ]:
# Pydantic models for POI search
class CategoryInfo(BaseModel):
    category: str
    group: str
    subCategory: str

class Address(BaseModel):
    city: str
    state: str
    countryCode: str
    streetName: str
    formattedAddress: str
    shortFormattedAddress: str
    zipCode: str

class RegionDetail(BaseModel):
    code: str
    name: str

class Regions(BaseModel):
    dma: RegionDetail
    state: RegionDetail
    cbsa: RegionDetail

class Venue(BaseModel):
    entityId: str
    entityType: str
    name: str
    categoryInfo: CategoryInfo
    address: Address
    isFlagged: bool
    regions: Regions
    apiId: str
    placerUrl: str
    storeId: Optional[str] = None
    isPermitted: bool

class PlacerResponse(BaseModel):
    data: List[Venue]
    requestId: str

print("✓ Pydantic models defined")

## Step 1: Find Reference POI

Search for nearby businesses at the target location to use as reference POI for demographics analysis.

In [ ]:
# Target POI coordinates
target_lat = 43.0859778
target_lng = -70.78677

# Search for ANY venue near the target location
target_search_url = "https://papi.placer.ai/v1/poi"
target_params = {
    "lat": target_lat,
    "lng": target_lng,
    "radius": 0.5,  # 0.5 miles
    "entityType": "venue",
    "limit": 20
}

headers = {
    "accept": "application/json",
    "x-api-key": placer_api_key
}

print(f"Searching for businesses near target location...\n")
print(f"Target: lat={target_lat}, lng={target_lng}, radius=0.5 miles\n")

response = requests.get(target_search_url, params=target_params, headers=headers)

print(f"Response status: {response.status_code}\n")

reference_poi_id = None
reference_poi_name = None

if response.status_code == 200:
    result = response.json()
    
    if result.get('data'):
        # Parse with Pydantic
        parsed_result = PlacerResponse.model_validate_json(response.text)
        
        print(f"Found {len(parsed_result.data)} nearby businesses:\n")
        print("="*80)
        
        for i, venue in enumerate(parsed_result.data[:10], 1):  # Show first 10
            print(f"{i}. {venue.name}")
            print(f"   Category: {venue.categoryInfo.category}")
            print(f"   Address: {venue.address.formattedAddress}")
            print(f"   API ID: {venue.apiId}")
            print()
        
        # Use the closest one (first in results) as reference
        if parsed_result.data:
            reference_poi_id = parsed_result.data[0].apiId
            reference_poi_name = parsed_result.data[0].name
            reference_category = parsed_result.data[0].categoryInfo.category
            print("="*80)
            print(f"\n✓ Using as reference POI: {reference_poi_name}")
            print(f"  Category: {reference_category}")
            print(f"  API ID: {reference_poi_id}\n")
        else:
            print("⚠ No businesses found nearby.\n")
    else:
        print("⚠ No data returned.\n")
else:
    print(f"Error: {response.text}")

if not reference_poi_id:
    print("\n⚠️  No nearby business found. Cannot proceed with demographics analysis.")

## Step 2: Pull Demographics for Drive Times 5-25 Minutes

Make API calls for each drive time from 5 to 25 minutes (1-minute increments) to get car parc data.

In [ ]:
# Calculate date range (last 5 months for demographics)
end_date = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=150)).strftime("%Y-%m-%d")

print(f"Date range for demographics: {start_date} to {end_date}")

In [ ]:
# Demographics API endpoint
demographics_url = "https://papi.placer.ai/v1/reports/trade-area-demographics"

headers_demographics = {
    "accept": "application/json",
    "content-type": "application/json",
    "x-api-key": placer_api_key
}

# Store results for each drive time
car_parc_data = []

print(f"\nFetching demographics for drive times 5-25 minutes...\n")
print(f"Reference POI: {reference_poi_name} ({reference_poi_id})\n")
print("="*80)

# Loop through drive times from 5 to 25 minutes
for drive_time in range(5, 26):
    print(f"Drive Time {drive_time} min: ", end="")
    
    payload = {
        "method": "driveTime",
        "benchmarkScope": "nationwide",
        "allocationType": "weightedCentroid",
        "trafficVolPct": 70,
        "withinRadius": drive_time,
        "ringRadius": 3,
        "dataset": "sti_popstats",
        "startDate": start_date,
        "endDate": end_date,
        "apiId": reference_poi_id,
        "driveTime": drive_time,
        "template": "default"
    }
    
    # Retry loop for IN_PROGRESS responses
    max_attempts = 15
    success = False
    
    for attempt in range(max_attempts):
        response = requests.post(demographics_url, json=payload, headers=headers_demographics)
        
        if response.status_code not in [200, 202]:
            print(f"✗ HTTP {response.status_code}")
            break
        
        # Check for IN_PROGRESS
        if "IN_PROGRESS" in response.text or response.status_code == 202:
            if attempt == 0:
                print("Processing", end="")
            print(".", end="", flush=True)
            time.sleep(2)
            continue
        
        # Success - extract car parc
        try:
            result = response.json()
            
            # Navigate to the vehicles data
            vehicles_data = result.get('data', {}).get('Vehicles per Household', {})
            total_vehicles = vehicles_data.get('Total Number of Vehicles', {}).get('value', 0)
            
            # Also get population and households for reference
            population = result.get('data', {}).get('Overview', {}).get('Population', {}).get('value', 0)
            households = result.get('data', {}).get('Overview', {}).get('Households', {}).get('value', 0)
            
            car_parc_data.append({
                'drive_time': drive_time,
                'car_parc': total_vehicles,
                'population': population,
                'households': households
            })
            
            print(f" ✓ Car Parc: {total_vehicles:,}")
            success = True
            break
            
        except Exception as e:
            print(f" ✗ Error: {str(e)[:50]}")
            break
    
    if not success:
        print(f"  (timeout after {max_attempts} attempts)")
    
    # Rate limiting - small delay between requests
    time.sleep(0.5)

print("\n" + "="*80)
print(f"\n✓ Completed: Retrieved car parc data for {len(car_parc_data)} drive times")

## Step 3: Display Results

In [ ]:
# Display results in table format
print("\n" + "="*80)
print("CAR PARC BY DRIVE TIME")
print("="*80 + "\n")

print(f"{'Drive Time':<15} {'Car Parc':>15} {'Population':>15} {'Households':>15}")
print("-" * 80)

for data in car_parc_data:
    print(f"{data['drive_time']} min{' ':<10} {data['car_parc']:>15,} {data['population']:>15,} {data['households']:>15,}")

print("-" * 80)

# Highlight the key drive times (10, 12, 15 minutes)
print("\n" + "="*80)
print("KEY DRIVE TIMES FOR ANALYSIS")
print("="*80 + "\n")

key_times = [10, 12, 15]
for drive_time in key_times:
    data_point = next((d for d in car_parc_data if d['drive_time'] == drive_time), None)
    if data_point:
        print(f"  {drive_time} min: {data_point['car_parc']:,} vehicles")

## Step 4: Calculate Business Metrics

Apply the business formulas:
- Total Addressable Market = Car Parc × Total Addressable Members %
- Washville Site Level Monthlies = Total Addressable Market × Washville Market Share %

In [ ]:
import numpy as np

# Known data points from the model
known_drive_times = np.array([10, 12, 15])
known_tam_pct = np.array([30, 25, 20])  # percentages
known_market_share_pct = np.array([95, 85, 65])  # percentages

# Interpolate/extrapolate for all drive times (5-25 minutes)
all_drive_times = np.array([d['drive_time'] for d in car_parc_data])

# Use linear interpolation (numpy.interp handles extrapolation too)
tam_percentages = np.interp(all_drive_times, known_drive_times, known_tam_pct)
market_share_percentages = np.interp(all_drive_times, known_drive_times, known_market_share_pct)

# Apply reasonable bounds
tam_percentages = np.clip(tam_percentages, 10, 40)  # Min 10%, Max 40%
market_share_percentages = np.clip(market_share_percentages, 40, 100)  # Min 40%, Max 100%

# Create lookup dictionaries
tam_pct_dict = {dt: tam_pct/100 for dt, tam_pct in zip(all_drive_times, tam_percentages)}
market_share_dict = {dt: ms_pct/100 for dt, ms_pct in zip(all_drive_times, market_share_percentages)}

print("\n" + "="*140)
print("INTERPOLATED TAM % AND MARKET SHARE % FOR ALL DRIVE TIMES")
print("="*140 + "\n")

print(f"{'Drive Time':<15} {'TAM %':>12} {'Market Share %':>18}")
print("-" * 50)

for dt, tam_pct, ms_pct in zip(all_drive_times, tam_percentages, market_share_percentages):
    marker = " *" if dt in [10, 12, 15] else ""
    print(f"{dt} min{' ':<10} {tam_pct:>11.1f}% {ms_pct:>17.1f}%{marker}")

print("-" * 50)
print("* = Known data points from model\n")

print("\n" + "="*140)
print("WASHVILLE MARKET ANALYSIS - ALL DRIVE TIMES")
print("="*140 + "\n")

print(f"{'Drive Time':<15} {'Car Parc':>15} {'TAM %':>10} {'TAM':>15} {'Mkt Share %':>15} {'Site Monthlies':>20}")
print("-" * 140)

for data_point in car_parc_data:
    drive_time = data_point['drive_time']
    car_parc = data_point['car_parc']
    tam_pct = tam_pct_dict[drive_time]
    market_share_pct = market_share_dict[drive_time]
    
    # Calculate metrics
    tam = int(car_parc * tam_pct)
    site_monthlies = int(tam * market_share_pct)
    
    marker = " *" if drive_time in [10, 12, 15] else ""
    print(f"{drive_time} min{' ':<10} {car_parc:>15,} {tam_pct*100:>9.1f}% {tam:>15,} {market_share_pct*100:>14.1f}% {site_monthlies:>20,}{marker}")

print("-" * 140)
print("* = Known data points from model\n")

# Highlight key drive times
print("\n" + "="*140)
print("KEY DRIVE TIMES (10, 12, 15 MINUTES)")
print("="*140 + "\n")

print(f"{'Drive Time':<15} {'Car Parc':>15} {'TAM %':>10} {'TAM':>15} {'Mkt Share %':>15} {'Site Monthlies':>20}")
print("-" * 140)

for drive_time in [10, 12, 15]:
    data_point = next((d for d in car_parc_data if d['drive_time'] == drive_time), None)
    
    if data_point:
        car_parc = data_point['car_parc']
        tam_pct = tam_pct_dict[drive_time]
        market_share_pct = market_share_dict[drive_time]
        
        # Calculate metrics
        tam = int(car_parc * tam_pct)
        site_monthlies = int(tam * market_share_pct)
        
        print(f"{drive_time} min{' ':<10} {car_parc:>15,} {tam_pct*100:>9.1f}% {tam:>15,} {market_share_pct*100:>14.1f}% {site_monthlies:>20,}")

print("-" * 140)

In [ ]:
# Create final formatted table matching the required output format - ALL DRIVE TIMES
print("\n" + "="*250)
print("FINAL OUTPUT - MARKET ANALYSIS SUMMARY (ALL DRIVE TIMES)")
print("="*250 + "\n")

print(f"Address: Portsmouth, NH (1303 Woodbury Ave)")
print(f"         1303 Woodbury Ave, Portsmouth, NH 03801\n")

# Get all available drive times
available_drive_times = sorted([d['drive_time'] for d in car_parc_data])

if len(car_parc_data) > 0:
    # Fixed column width for each drive time
    col_width = 12
    
    # Build header with all drive times
    header_row = f"{'':40}"  # Left column for row labels
    for dt in available_drive_times:
        header_row += f"{dt:>{col_width-4}} Min"
    print(header_row)
    print(f"{'':40}" + "=" * (len(available_drive_times) * col_width))
    
    # Total Car Parc
    car_parc_row = f"{'Total Car Parc':<40}"
    for dt in available_drive_times:
        data = next((d for d in car_parc_data if d['drive_time'] == dt), None)
        if data:
            car_parc_row += f"{data['car_parc']:>{col_width},}"
        else:
            car_parc_row += f"{'N/A':>{col_width}}"
    print(car_parc_row)
    print()  # Blank line
    
    # Total Addressable Members %
    tam_pct_row = f"{'Total Addressable Members':<40}"
    for dt in available_drive_times:
        tam_pct = tam_pct_dict.get(dt, 0)
        tam_pct_row += f"{tam_pct*100:>{col_width-1}.1f}%"
    print(tam_pct_row)
    
    # Total Addressable Market
    tam_row = f"{'Total Addressable Market':<40}"
    for dt in available_drive_times:
        data = next((d for d in car_parc_data if d['drive_time'] == dt), None)
        if data:
            car_parc = data['car_parc']
            tam_pct = tam_pct_dict.get(dt, 0)
            tam = int(car_parc * tam_pct)
            tam_row += f"{tam:>{col_width},}"
        else:
            tam_row += f"{'N/A':>{col_width}}"
    print(tam_row)
    print()  # Blank line
    
    # Washville Market Share %
    market_share_row = f"{'Washville Market Share':<40}"
    for dt in available_drive_times:
        market_share_pct = market_share_dict.get(dt, 0)
        market_share_row += f"{market_share_pct*100:>{col_width-1}.1f}%"
    print(market_share_row)
    
    # Washville Site Level Monthlies
    monthlies_row = f"{'Washville Site Level Monthlies':<40}"
    for dt in available_drive_times:
        data = next((d for d in car_parc_data if d['drive_time'] == dt), None)
        if data:
            car_parc = data['car_parc']
            tam_pct = tam_pct_dict.get(dt, 0)
            market_share_pct = market_share_dict.get(dt, 0)
            tam = int(car_parc * tam_pct)
            site_monthlies = int(tam * market_share_pct)
            monthlies_row += f"{site_monthlies:>{col_width},}"
        else:
            monthlies_row += f"{'N/A':>{col_width}}"
    print(monthlies_row)
    
    print("\n" + "="*250)
    
    # Add legend
    print("\nNote:")
    print("  - TAM % and Market Share % are interpolated for drive times outside 10, 12, 15 minutes")
else:
    print("⚠ No car parc data available")